In [2]:
from IPython.display import Image

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, END
from langchain.prompts import ChatPromptTemplate
from pymongo import MongoClient
from dotenv import load_dotenv
import os
import json
import traceback
from datetime import datetime, timedelta
from typing import TypedDict, List

# -------------------- CONFIG --------------------
load_dotenv()

GEMINI_API_KEY = os.getenv("GOOGLE_API_KEY")
MONGO_URI = os.getenv("MONGO_URI")

if not GEMINI_API_KEY:
    raise ValueError("❌ GOOGLE_API_KEY not found in .env")
if not MONGO_URI:
    raise ValueError("❌ MONGO_URI not found in .env")

llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", api_key=GEMINI_API_KEY)
client = MongoClient(MONGO_URI)
expenses_collection = client["expense_db"]["expenses"]

# -------------------- DEBUG HELPER --------------------
DEBUG = True

def debug_print(msg: str, obj=None):
    """Pretty-print debug messages with timestamp."""
    if not DEBUG:
        return
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"\n[DEBUG {ts}] {msg}")
    if obj is not None:
        try:
            # try to pretty-print json-like objects
            print(json.dumps(obj, indent=2, default=str))
        except Exception:
            # fallback to repr
            print(repr(obj))

# -------------------- STATE --------------------
# --- In the STATE section of your code ---

class ConversationState(TypedDict):
    conversation_history: List[str]
    last_intent: str
    current_entities: dict
    response: dict
    
    # --- NEW FIELDS FOR THE ANALYTICS LOOP ---
    user_question: str          # The original analytics question from the user.
    query_plan: dict            # The structured query plan generated by the LLM.
    query_results: List[dict]   # The data retrieved from MongoDB.
    feedback_message: str       # A message to inform the planner what went wrong.
    iteration_count: int        # The counter for our loop.

# -------------------- TOOLS --------------------

def extract_expense_entities(user_input: str, llm: ChatGoogleGenerativeAI) -> dict:
    """
    Uses Gemini to extract structured expense data from a user's natural language query.
    """
    debug_print("extract_expense_entities() called with user_input:", user_input)
    today = datetime.now().strftime("%Y-%m-%d")
    yesterday = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")

    prompt = ChatPromptTemplate.from_template(
        """
        You are an expert entity extraction system for an expense tracker.
        Your task is to extract expense details from the user query and return a single, valid JSON object.
        Do NOT include any text or markdown formatting before or after the JSON object.

        Context:
        - Today's date is {date}.
        - Yesterday's date is {yesterday}.

        JSON Schema:
        {{
          "amount": <number, required>,
          "category": <string, required: "food", "transport", "shopping", "medication", "entertainment", "utilities", "other">,
          "description": <string, required: a brief summary of the expense>,
          "people": <array of strings, optional: names of people involved>,
          "date": <string, required: in "YYYY-MM-DD" format>
        }}

        Examples:
        1. Query: "I spent Rs 500 on medication with my friend Ajay"
           Output: {{"amount": 500, "category": "medication", "description": "Medication purchase", "people": ["Ajay"], "date": "{date}"}}
        2. Query: "yesterday we had dinner for 2,500"
           Output: {{"amount": 2500, "category": "food", "description": "Dinner", "people": [], "date": "{yesterday}"}}
        3. Query: "flight tickets to delhi cost 12000"
           Output: {{"amount": 12000, "category": "transport", "description": "Flight tickets to Delhi", "people": [], "date": "{date}"}}
        ---
        User Query: "{query}"
        """
    )

    chain = prompt | llm
    try:
        response = chain.invoke({"date": today, "yesterday": yesterday, "query": user_input})
        content = getattr(response, "content", str(response))
        cleaned_content = content.strip().replace("```json", "").replace("```", "").strip()
        entities = json.loads(cleaned_content)
        entities["raw_text"] = user_input
        debug_print("extract_expense_entities() successfully extracted:", entities)
        return entities
    except (json.JSONDecodeError, Exception) as e:
        debug_print("extract_expense_entities() failed:", str(e))
        return {
            "amount": None, "category": "uncategorized", "description": "Failed to parse details",
            "people": [], "date": today, "raw_text": user_input, "error": str(e)
        }


def classify_intent(user_input: str) -> str:
    """Classify intent into add_expense, analytics, or chit_chat"""
    debug_print("classify_intent() called with user_input:", user_input)
    prompt = ChatPromptTemplate.from_template(
        "Classify this user query: {query}. "
        "Return only one label exactly: add_expense | analytics | chit_chat"
    )
    try:
        response = llm.invoke(prompt.format(query=user_input))
        content = getattr(response, "content", str(response)).strip().lower()
        if content not in ["add_expense", "analytics", "chit_chat"]:
            debug_print("classify_intent() produced unexpected label, falling back.", content)
            return "chit_chat"
        debug_print("classify_intent() final intent:", content)
        return content
    except Exception as e:
        debug_print("classify_intent() LLM call failed:", str(e))
        return "chit_chat"  # safe fallback

def add_expense_tool(entities: dict) -> dict:
    """Insert parsed expense into MongoDB"""
    debug_print("add_expense_tool() called with entities:", entities)
    try:
        res = expenses_collection.insert_one(entities)
        result = {"status": "ok", "inserted_id": str(res.inserted_id), "entities": entities}
        debug_print("add_expense_tool() insertion result:", result)
        return result
    except Exception as e:
        debug_print("add_expense_tool() failed:", str(e))
        return {"status": "error", "error": str(e)}

def analytics_tool(query: str) -> dict:
    """Query MongoDB and return insights (example pipeline)."""
    debug_print("analytics_tool() called with query:", query)
    try:
        # NOTE: This is a placeholder. You will build pipelines based on parsed entities.
        pipeline = [
            {"$match": {"category": "food"}},
            {"$group": {"_id": None, "total": {"$sum": "$amount"}}}
        ]
        result = list(expenses_collection.aggregate(pipeline))
        if result:
            return {"status": "ok", "total": result[0]["total"]}
        return {"status": "ok", "total": 0}
    except Exception as e:
        debug_print("analytics_tool() failed:", str(e))
        return {"status": "error", "error": str(e)}

def chit_chat_tool(user_input: str) -> dict:
    """General chit chat handled by Gemini"""
    debug_print("chit_chat_tool() calling LLM with:", user_input)
    try:
        response = llm.invoke(user_input)
        content = getattr(response, "content", str(response))
        return {"status": "ok", "reply": content}
    except Exception as e:
        debug_print("chit_chat_tool() LLM call failed:", str(e))
        return {"status": "error", "error": str(e)}



def create_analytics_plan(question: str, history: List[str], feedback: str, llm: ChatGoogleGenerativeAI) -> dict:
    """Uses Gemini to create a structured MongoDB query plan from a user's question."""
    debug_print("create_analytics_plan() called with question:", question)
    
    # Provide more context by including the last few messages
    context = "\n".join(history[-5:])

    prompt = ChatPromptTemplate.from_template(
        """
        You are an expert data analyst. Your task is to create a structured JSON plan to query a MongoDB database
        based on a user's question. Do NOT include any text or markdown formatting around the JSON object.

        **Context of the conversation:**
        {context}

        **Previous attempt feedback (if any):**
        {feedback}

        **Database Schema:**
        Expenses are stored in a collection with documents like:
        {{ "amount": 500, "category": "medication", "description": "...", "people": ["Ajay"], "date": "2025-09-03" }}

        **JSON Plan Schema:**
        {{
          "query_type": <"SUM" | "GROUP_BY" | "COUNT">,
          "filters": {{
            "category": <string, optional>,
            "date_range": {{ "start": "YYYY-MM-DD", "end": "YYYY-MM-DD" }}, optional
            "people": <array of strings, optional>
          }},
          "group_by_field": <"category" | "people", optional, required if query_type is "GROUP_BY">,
          "visualization_hint": <"total" | "pie_chart" | "bar_chart" | "list">
        }}

        **Examples:**
        1. Question: "how much did I spend on food last month?"
           Output: {{"query_type": "SUM", "filters": {{"category": "food", "date_range": {{"start": "2025-08-01", "end": "2025-08-31"}}}}, "visualization_hint": "total"}}
        2. Question: "show my spending breakdown by category"
           Output: {{"query_type": "GROUP_BY", "filters": {{}}, "group_by_field": "category", "visualization_hint": "pie_chart"}}
        ---
        User Question: "{question}"
        """
    )
    chain = prompt | llm
    try:
        response = chain.invoke({"question": question, "context": context, "feedback": feedback or "None"})
        content = getattr(response, "content", str(response))
        cleaned_content = content.strip().replace("```json", "").replace("```", "").strip()
        plan = json.loads(cleaned_content)
        debug_print("create_analytics_plan() successfully generated plan:", plan)
        return plan
    except Exception as e:
        debug_print("create_analytics_plan() failed:", str(e))
        return {"error": "Failed to generate a valid query plan."}

def execute_mongo_pipeline(plan: dict) -> dict:
    """Dynamically builds and executes a MongoDB aggregation pipeline from a plan."""
    debug_print("execute_mongo_pipeline() called with plan:", plan)
    if "error" in plan:
        return {"status": "error", "error": "Invalid plan received."}
    
    try:
        # --- Build the $match stage from filters ---
        match_stage = {}
        filters = plan.get("filters", {})
        if filters.get("category"):
            match_stage["category"] = filters["category"]
        if filters.get("date_range"):
            match_stage["date"] = {
                "$gte": filters["date_range"]["start"],
                "$lte": filters["date_range"]["end"]
            }
        
        pipeline = [{"$match": match_stage}] if match_stage else []

        # --- Build the $group and $project stages ---
        query_type = plan.get("query_type")
        if query_type == "SUM":
            pipeline.append({"$group": {"_id": None, "total": {"$sum": "$amount"}}})
        elif query_type == "GROUP_BY":
            group_field = plan.get("group_by_field")
            if not group_field:
                return {"status": "error", "error": "GROUP_BY requires a group_by_field."}
            pipeline.append({"$group": {"_id": f"${group_field}", "total": {"$sum": "$amount"}}})
            pipeline.append({"$project": {"name": "$_id", "total": 1, "_id": 0}})

        debug_print("execute_mongo_pipeline() constructed pipeline:", pipeline)
        results = list(expenses_collection.aggregate(pipeline))
        debug_print("execute_mongo_pipeline() got results:", results)
        return {"status": "ok", "results": results}

    except Exception as e:
        debug_print("execute_mongo_pipeline() failed:", str(e))
        return {"status": "error", "error": str(e)}

# -------------------- NODES --------------------
def intent_classifier(state: ConversationState):
    debug_print("Entering node: intent_classifier")
    user_input = state["conversation_history"][-1]
    intent = classify_intent(user_input)
    state["last_intent"] = intent
    debug_print("Exiting node: intent_classifier", {"last_intent": state["last_intent"]})
    return state

    def add_expense_node(state: ConversationState):
        debug_print("Entering node: add_expense_node")
        user_input = state["conversation_history"][-1]

        # 1. Call Gemini-based extractor
        entities = extract_expense_entities(user_input, llm)
        state["current_entities"] = entities

        # 2. Basic validation
        if not entities.get("amount"):
            state["response"] = {
                "message": "I see you want to add an expense, but I couldn't determine the amount. Could you please clarify?",
                "details": entities
            }
            debug_print("Exiting node: add_expense_node (amount missing)", state["response"])
            return state

        # 3. Call database tool
        tool_result = add_expense_tool(entities)

        # 4. Formulate response
        if tool_result.get('status') == 'ok':
            message = f"✅ Logged! Expense of {entities['amount']} for '{entities['description']}' in the {entities['category']} category."
        else:
            message = f"❌ Sorry, I failed to save the expense. Error: {tool_result.get('error')}"

        state["response"] = {"message": message, "details": tool_result}
        debug_print("Exiting node: add_expense_node", state["response"])
        return state

def analytics_node(state: ConversationState):
    debug_print("Entering node: analytics_node")
    user_input = state["conversation_history"][-1]
    # TODO: Use entity extraction to build exact aggregation pipeline
    tool_result = analytics_tool(user_input)
    if tool_result.get("status") == "ok":
        state["response"] = {"message": f"Total spend on food so far is: {tool_result.get('total')}", "result": tool_result}
    else:
        state["response"] = {"message": "Analytics failed", "error": tool_result.get("error")}
    debug_print("Exiting node: analytics_node", state["response"])
    return state

def chit_chat_node(state: ConversationState):
    debug_print("Entering node: chit_chat_node")
    user_input = state["conversation_history"][-1]
    tool_result = chit_chat_tool(user_input)
    if tool_result.get("status") == "ok":
        state["response"] = {"message": tool_result.get("reply")}
    else:
        state["response"] = {"message": "Sorry, I couldn't get a response.", "error": tool_result.get("error")}
    debug_print("Exiting node: chit_chat_node", state["response"])
    return state


def analytics_planner_node(state: ConversationState):
    """Generates a structured plan for how to query the database."""
    debug_print("Entering node: analytics_planner_node")
    state["iteration_count"] += 1
    
    # On the first iteration, the user question is the latest message.
    if state["iteration_count"] == 1:
        state["user_question"] = state["conversation_history"][-1]

    plan = create_analytics_plan(
        question=state["user_question"],
        history=state["conversation_history"],
        feedback=state["feedback_message"],
        llm=llm
    )
    state["query_plan"] = plan
    debug_print("Exiting node: analytics_planner_node", {"plan": plan, "iteration": state["iteration_count"]})
    return state

def query_mongodb_node(state: ConversationState):
    """Executes the query plan against MongoDB."""
    debug_print("Entering node: query_mongodb_node")
    tool_result = execute_mongo_pipeline(state["query_plan"])
    
    if tool_result["status"] == "ok" and not tool_result["results"]:
        state["feedback_message"] = "The query returned no data. The user's request might be for a time period with no expenses or for a non-existent category. Try asking for clarification."
    else:
        state["feedback_message"] = "" # Clear feedback on success
        
    state["query_results"] = tool_result.get("results", [])
    debug_print("Exiting node: query_mongodb_node", {"results_count": len(state["query_results"])})
    return state

def summarize_results_node(state: ConversationState):
    """Generates a final, user-friendly response based on the query results."""
    debug_print("Entering node: summarize_results_node")
    prompt = ChatPromptTemplate.from_template(
        """You are a helpful financial assistant.
        The user asked: "{question}"
        The database returned the following data: {data}
        
        Please provide a concise, friendly, and clear answer to the user's question based on the data.
        """
    )
    chain = prompt | llm
    response = chain.invoke({"question": state["user_question"], "data": json.dumps(state["query_results"])})
    
    state["response"] = {"message": getattr(response, "content", str(response))}
    debug_print("Exiting node: summarize_results_node", state["response"])
    return state
# -------------------- GRAPH --------------------
def should_continue_or_end(state: ConversationState):
    debug_print("Entering conditional edge: should_continue_or_end")
    if state["iteration_count"] >= 5:
        debug_print("Decision: Max iterations reached. Ending.")
        # Optional: You could create a specific "failure_node" to handle this
        state["response"] = {"message": "Sorry, I tried a few times but couldn't figure out how to answer that. Could you rephrase your question?"}
        return "end"
    
    if state["feedback_message"]: # If there's feedback, it means the query failed or returned no data
        debug_print("Decision: Feedback found. Continuing loop.")
        return "continue"
    
    debug_print("Decision: Success. Summarizing results.")
    return "summarize"

graph = StateGraph(ConversationState)

graph.add_node("intent_classifier", intent_classifier)
graph.add_node("add_expense", add_expense_node)
# graph.add_node("analytics", analytics_node)
graph.add_node("chit_chat", chit_chat_node)
graph.add_node("analytics_planner", analytics_planner_node)
graph.add_node("query_mongodb", query_mongodb_node)
graph.add_node("summarize_results", summarize_results_node)


# This is the start of making the graph
graph.set_entry_point("intent_classifier")

# Original conditional edges from the classifier
graph.add_conditional_edges(
    "intent_classifier",
    lambda state: state["last_intent"],
    {
        "add_expense": "add_expense",
        "analytics": "analytics_planner", # <-- Route analytics intent to our new planner
        "chit_chat": "chit_chat",
    }
)
# Define the new analytics loop
graph.add_edge("analytics_planner", "query_mongodb")
graph.add_conditional_edges(
    "query_mongodb",
    should_continue_or_end,
    {
        "continue": "analytics_planner", # <-- On failure, go back to the planner
        "summarize": "summarize_results", # <-- On success, go to the summarizer
        "end": END
    }
)


# Define the end points for all paths
graph.add_edge("add_expense", END)
graph.add_edge("chit_chat", END)
graph.add_edge("summarize_results", END)

app = graph.compile()



ConfigurationError: All nameservers failed to answer the query cluster0.zvyfclc.mongodb.net. IN TXT: Server Do53:172.16.1.2@53 answered The DNS operation timed out.; Server Do53:192.168.137.1@53 answered The DNS operation timed out.; Server Do53:172.16.1.2@53 answered The DNS operation timed out.; Server Do53:192.168.137.1@53 answered The DNS operation timed out.; Server Do53:172.16.1.2@53 answered [WinError 10051] A socket operation was attempted to an unreachable network; Server Do53:192.168.137.1@53 answered [WinError 10051] A socket operation was attempted to an unreachable network

In [29]:
print (app)

In [30]:
app.get_graph()

Graph(nodes={'__start__': Node(id='__start__', name='__start__', data=RunnableCallable(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), 'intent_classifier': Node(id='intent_classifier', name='intent_classifier', data=intent_classifier(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), 'add_expense': Node(id='add_expense', name='add_expense', data=add_expense(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), 'chit_chat': Node(id='chit_chat', name='chit_chat', data=chit_chat(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), 'analytics_planner': Node(id='analytics_planner', name='analytics_planner', data=analytics_planner(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), 'query_mongodb': Node(id='query_mongodb', name='query_mongodb', data=query_mongodb(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), 'summarize_resul

In [31]:
print(app.get_graph().draw_ascii())

                                                +-----------+                                            
                                                | __start__ |                                            
                                                +-----------+                                            
                                                      *                                                  
                                                      *                                                  
                                                      *                                                  
                                            +-------------------+                                        
                                            | intent_classifier |.                                       
                                           .+-------------------+ .......                                
                                      .....   